# 09 · Red Neuronal Convolucional (CNN) con CIFAR-10
**Curso ITEC · C-DAC Noida — Programa Especializado en IA**

En este cuaderno comparamos una **red neuronal normal (ANN)** contra una **red convolucional (CNN)**
clasificando imágenes a color del dataset **CIFAR-10** (10 tipos de objetos).
La idea es ver, con las mismas fotos, por qué la CNN es muy superior para imágenes.

Todo corre en Colab sin subir nada: el dataset ya viene dentro de Keras.

### 1. Instalar e importar las herramientas
TensorFlow es el motor de deep learning; Keras es la interfaz amigable encima de él.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
print('TensorFlow', tf.__version__)

### 2. Cargar el dataset CIFAR-10
Son **60.000 imágenes a color de 32×32 píxeles** en 10 clases.
`load_data()` las trae ya divididas en entrenamiento (50.000) y prueba (10.000).

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
print('X_train:', X_train.shape)   # (50000, 32, 32, 3) -> 50000 fotos de 32x32 a color (RGB)
print('X_test :', X_test.shape)

Las etiquetas vienen como columna (2D). Para clasificar nos basta una lista (1D), así que las aplanamos.

In [ ]:
y_train = y_train.reshape(-1,)
y_test  = y_test.reshape(-1,)
clases = ['avión','auto','pájaro','gato','ciervo','perro','rana','caballo','barco','camión']
print(y_train[:5])

### 3. Ver algunas imágenes
Así confirmamos qué contiene el dataset.

In [ ]:
def ver(X, y, i):
    plt.figure(figsize=(2,2))
    plt.imshow(X[i])
    plt.xlabel(clases[y[i]])
    plt.xticks([]); plt.yticks([])
    plt.show()

ver(X_train, y_train, 0)
ver(X_train, y_train, 1)

### 4. Normalizar (muy importante)
Cada píxel va de 0 a 255. Dividimos entre 255 para dejarlo en el rango **0 a 1**.
Con valores pequeños la red aprende de forma más estable y rápida.

In [ ]:
X_train = X_train / 255.0
X_test  = X_test  / 255.0

### 5. Primero una red NORMAL (ANN)
Aplana la imagen de una vez y usa solo capas densas. Veremos que le cuesta con fotos a color.

In [ ]:
ann = keras.models.Sequential([
    keras.layers.Flatten(input_shape=(32,32,3)),
    keras.layers.Dense(3000, activation='relu'),
    keras.layers.Dense(1000, activation='relu'),
    keras.layers.Dense(10,   activation='softmax')   # 10 clases, softmax = una sola respuesta
])
ann.compile(optimizer='SGD',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])
ann.fit(X_train, y_train, epochs=5)

La ANN termina alrededor del **~49%** de acierto. Flojo para imágenes.

In [ ]:
from sklearn.metrics import classification_report
y_pred = ann.predict(X_test)
y_pred_clases = [np.argmax(p) for p in y_pred]
print(classification_report(y_test, y_pred_clases, target_names=clases))

### 6. Ahora una CNN (red convolucional)
Aquí está la diferencia:
- **Conv2D**: filtros que detectan formas (bordes, curvas) recorriendo la imagen.
- **MaxPooling2D**: encoge el mapa conservando lo importante (menos cálculo, tolera que el objeto esté movido).
- **Flatten**: aplana a 1D para la parte que decide.
- **Dense + softmax**: la red que clasifica en las 10 clases.

In [ ]:
cnn = keras.models.Sequential([
    keras.layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu', input_shape=(32,32,3)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])
cnn.compile(optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])
cnn.fit(X_train, y_train, epochs=5)

La CNN sube a **~70%** con las mismas fotos: más de 20 puntos por encima de la ANN,
y con menos cálculo porque el pooling reduce el tamaño. **Las CNN son las mejores para imágenes.**

In [ ]:
cnn.evaluate(X_test, y_test)

### 7. Probar una predicción
Tomamos una imagen de prueba y vemos qué dice la CNN.

In [ ]:
y_pred = cnn.predict(X_test)
y_clases = [np.argmax(p) for p in y_pred]

ver(X_test, y_test, 3)                 # la imagen real
print('La CNN dice que es:', clases[y_clases[3]])

---
### Resumen
| Red | Precisión (5 épocas) |
|-----|----------------------|
| **ANN** (red normal) | ~49% |
| **CNN** (convolucional) | ~70% |

La CNN primero **VE** (convolución + ReLU + pooling) y luego **DECIDE** (flatten + densas).
Por eso gana claramente en imágenes.

*Núcleo IA · curso-itec-c-dac.web.app · C-DAC Noida (ITEC) · 2026*